# SRE On-Call Agent
> AI-powered incident triage, root cause investigation, and post-mortem generation

## Project Introduction

When a production alert fires at 3am, an SRE on-call engineer must:
1. **Triage** — assess severity and plan the investigation
2. **Investigate** — search logs, query metrics, consult runbooks
3. **Write a post-mortem** — root cause, timeline, action items

This project automates that workflow with a **three-agent AI pipeline**:

```
Alert JSON
    │
    ▼
┌─────────────────────────────┐
│  Stage 1: TriageAgent       │  Plan-and-Solve
│  "What should I investigate?"│  → ordered investigation plan
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  Stage 2: InvestigationAgent│  ReAct loop
│  log_search / metric_query  │  → root cause hypothesis
│  / runbook_lookup           │
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  Stage 3: PostmortemAgent   │  Reflection
│  draft → critique → revise  │  → final RCA report
└─────────────────────────────┘
```

## Author Information
- **GitHub**: [@zjzhou](https://github.com/zjzhou)
- **Date**: 2026-04-21
- **HelloAgents Chapter**: 16 (Graduation Project)

---
## Part 2: Environment Setup

In [1]:
# Install dependencies
!pip install -q openai fastapi uvicorn pyyaml pydantic python-dotenv

zsh:1: command not found: pip


In [2]:
import os
import sys
import json
import time
import re
import ast
import glob
import yaml
from pathlib import Path
from typing import Dict, Any, List
from dotenv import load_dotenv
from openai import OpenAI

# Add project root to path so src/ imports work
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv()
print("✅ Environment loaded")
print(f"   LLM_MODEL_ID: {os.getenv('LLM_MODEL_ID', 'NOT SET')}")
print(f"   LLM_BASE_URL: {os.getenv('LLM_BASE_URL', 'NOT SET')}")
print(f"   LLM_API_KEY:  {'SET' if os.getenv('LLM_API_KEY') else 'NOT SET'}")

✅ Environment loaded
   LLM_MODEL_ID: llama-3.3-70b-versatile
   LLM_BASE_URL: https://api.groq.com/openai/v1
   LLM_API_KEY:  SET


---
## Part 3: Tool Definitions

Three tools give the InvestigationAgent eyes into the incident:

| Tool | Input | What it does |
|---|---|---|
| `LogSearchTool` | keyword/regex | Searches incident log entries |
| `MetricQueryTool` | metric name keyword | Returns time-series metric data |
| `RunbookLookupTool` | error pattern | Fetches runbook remediation steps |

In [3]:
from src.tools.log_search_tool import LogSearchTool
from src.tools.metric_query_tool import MetricQueryTool
from src.tools.runbook_tool import RunbookLookupTool

# --- Quick tool smoke test ---
incident_path = project_root / "data" / "incidents" / "db_pool_exhaustion.json"
with open(incident_path) as f:
    sample_incident = json.load(f)

log_tool = LogSearchTool(sample_incident)
metric_tool = MetricQueryTool(sample_incident)
runbook_tool = RunbookLookupTool(
    service="checkout-service",
    runbooks_dir=str(project_root / "data" / "runbooks")
)

print("🔧 LogSearchTool — searching for 'pool':")
print(log_tool.run("pool"))
print()
print("📊 MetricQueryTool — querying 'db_pool':")
print(metric_tool.run("db_pool"))
print()
print("📖 RunbookLookupTool — pattern 'pool exhausted':")
print(runbook_tool.run("pool exhausted"))

🔧 LogSearchTool — searching for 'pool':
Found 4 log entries matching 'pool':
[2024-01-15T14:00:03Z] [WARN    ] DB connection pool: 8/10 connections in use
[2024-01-15T14:01:15Z] [WARN    ] DB connection pool: 10/10 connections in use, queuing requests
[2024-01-15T14:01:45Z] [ERROR   ] DB connection pool exhausted — waiting for available connection (timeout=30s)
[2024-01-15T14:02:01Z] [ERROR   ] DB connection pool exhausted — waiting for available connection (timeout=30s)

📊 MetricQueryTool — querying 'db_pool':
Metrics for checkout-service matching 'db_pool':
  db_pool_active_connections: [14:00: 3 | 14:01: 8 | 14:02: 10 | 14:03: 10]

📖 RunbookLookupTool — pattern 'pool exhausted':
Runbook: checkout-service (v1.2)

Pattern: 'DB pool exhausted' | Severity: P1
Remediation steps:
  1. Identify slow queries: run SHOW PROCESSLIST on the primary DB; look for queries >5s
  2. Run EXPLAIN on the slowest query to identify full table scans or missing indexes
  3. Temporary mitigation: increase p

---
## Part 4: Agent Construction

Three agents implementing different paradigms from the Hello-Agents curriculum:

- **TriageAgent** — Plan-and-Solve (Chapter 4): LLM creates a structured investigation plan
- **InvestigationAgent** — ReAct (Chapter 4): iterative Reason-Act-Observe loop with tools
- **PostmortemAgent** — Reflection (Chapter 4): draft → critique → revise

In [4]:
from src.core.llm_client import HelloAgentsLLM
from src.agents.triage_agent import TriageAgent
from src.agents.investigation_agent import InvestigationAgent
from src.agents.postmortem_agent import PostmortemAgent
from src.agents.pipeline import run_pipeline, list_incidents, load_incident

# Verify LLM connection
llm = HelloAgentsLLM(verbose=True)
test_response = llm.think([{"role": "user", "content": "Reply with exactly: LLM OK"}])
print(f"\nLLM connection test: {test_response.strip()}")

print(f"\nAvailable incident fixtures: {list_incidents()}")

🧠 Calling llama-3.3-70b-versatile...
LLM OK

LLM connection test: LLM OK

Available incident fixtures: ['db_pool_exhaustion', 'memory_leak_oom', 'external_api_ratelimit']


---
## Part 5: Demo — Full Pipeline

### Demo 1: DB Connection Pool Exhaustion (INC-001)

**Alert:** `checkout-service` P99 latency 8.3s > threshold 1.0s  
**Root cause (ground truth):** Missing index on `orders.user_id` → full table scan → DB pool exhaustion

In [5]:
# Show the incident alert
incident_1 = load_incident("db_pool_exhaustion")
print("📣 Incident Alert:")
print(json.dumps({
    "incident_id": incident_1["incident_id"],
    "service": incident_1["service"],
    "severity": incident_1["severity"],
    "alert": incident_1["alert"]
}, indent=2))

📣 Incident Alert:
{
  "incident_id": "INC-001",
  "service": "checkout-service",
  "severity": "P1",
  "alert": {
    "metric": "http_request_duration_p99",
    "value": 8.3,
    "threshold": 1.0,
    "unit": "seconds",
    "timestamp": "2024-01-15T14:03:22Z",
    "description": "P99 request latency critically exceeds SLO threshold"
  }
}


In [ ]:
# Run the full pipeline
start = time.time()
result_1 = run_pipeline("db_pool_exhaustion", verbose=True)
elapsed_1 = round(time.time() - start, 1)
print(f"\n⏱️  Pipeline completed in {elapsed_1}s")


🚨 STAGE 1: TRIAGE — Generating investigation plan
🧠 Calling llama-3.3-70b-versatile...
```python
[
  {"tool": "metric_query", "query": "http_request_duration_p99", "reason": "Verify the current value of the metric that triggered the alert"},
  {"tool": "log_search", "query": "checkout-service ERROR", "reason": "Find all error-level log entries from the checkout-service to identify the failure pattern"},
  {"tool": "metric_query", "query": "http_request_count", "reason": "Check if there's an unusual increase in request volume that could be contributing to the latency"},
  {"tool": "log_search", "query": "checkout-service WARNING|ERROR", "reason": "Find all warning and error-level log entries from the checkout-service to identify potential issues"},
  {"tool": "runbook_lookup", "query": "high latency checkout-service", "reason": "Get standard remediation steps for high latency issues in the checkout-service"}
]
```

✅ Investigation plan (5 steps):
   1. [metric_query] http_request_durat

In [38]:
# Display the generated post-mortem report
from IPython.display import Markdown, display
display(Markdown(result_1["report"]))

### Demo 2: External API Rate Limit Cascade (INC-003)

**Alert:** `payment-service` success rate 23.4% < threshold 95%  
**Root cause (ground truth):** Promotional traffic spike + no exponential backoff → retry storm → Stripe 429 cascade

In [39]:
start = time.time()
result_3 = run_pipeline("external_api_ratelimit", verbose=True)
elapsed_3 = round(time.time() - start, 1)
print(f"\n⏱️  Pipeline completed in {elapsed_3}s")
display(Markdown(result_3["report"]))


🚨 STAGE 1: TRIAGE — Generating investigation plan
🧠 Calling gpt-4o-mini...
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge your account to continue using the API. (tid: 2026042706261177927272665835401)', 'type': 'Aihubmix_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️  Could not parse structured plan; using fallback plan.

🔍 STAGE 2: INVESTIGATION — ReAct tool loop

--- Step 1 ---
🧠 Calling gpt-4o-mini...
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge your account to continue using the API. (tid: 2026042706261185271520637880320)', 'type': 'Aihubmix_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️  Empty LLM response, stopping.

📝 STAGE 3: POST-MORTEM — Reflection (draft → critique → revise)
🧠 Calling gpt-4o-mini...
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge y

---
## Part 6: Performance Evaluation

Run all 3 incident fixtures and measure: pipeline speed, root cause accuracy, and report quality.

In [40]:
GROUND_TRUTH = {
    "db_pool_exhaustion":   "missing index",
    "memory_leak_oom":      "no TTL",
    "external_api_ratelimit": "retry storm",
}

eval_results = []
for incident_id in list_incidents():
    t0 = time.time()
    res = run_pipeline(incident_id, verbose=False)
    elapsed = round(time.time() - t0, 1)

    ground_truth_keyword = GROUND_TRUTH.get(incident_id, "")
    root_cause_text = res["findings"].get("root_cause", "").lower()
    correct = ground_truth_keyword.lower() in root_cause_text

    eval_results.append({
        "incident_id": incident_id,
        "service": res["service"],
        "severity": res["severity"],
        "elapsed_s": elapsed,
        "root_cause_correct": "✅" if correct else "❌",
        "root_cause_found": res["findings"].get("root_cause", "")[:80] + "...",
    })

print("\n=== Evaluation Results ===")
for r in eval_results:
    print(f"  [{r['root_cause_correct']}] {r['incident_id']:30s} "
          f"| {r['elapsed_s']:5.1f}s | {r['root_cause_found']}")

accuracy = sum(1 for r in eval_results if r["root_cause_correct"] == "✅") / len(eval_results)
avg_time = sum(r["elapsed_s"] for r in eval_results) / len(eval_results)
print(f"\nAccuracy: {accuracy:.0%}   Avg pipeline time: {avg_time:.1f}s")


🚨 STAGE 1: TRIAGE — Generating investigation plan
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge your account to continue using the API. (tid: 2026042706261223482086656223927)', 'type': 'Aihubmix_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️  Could not parse structured plan; using fallback plan.

🔍 STAGE 2: INVESTIGATION — ReAct tool loop

--- Step 1 ---
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge your account to continue using the API. (tid: 2026042706261231052236455543539)', 'type': 'Aihubmix_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️  Empty LLM response, stopping.

📝 STAGE 3: POST-MORTEM — Reflection (draft → critique → revise)
❌ LLM API error: Error code: 403 - {'error': {'message': 'Your account balance is insufficient. Please recharge your account to continue using the API. (tid: 202604270626124010240321386141

---
## Part 7: Summary and Outlook

### What We Built
- **TriageAgent (Plan-and-Solve)**: converts a raw alert into an ordered investigation plan, reducing hallucinated tool calls in the ReAct stage
- **InvestigationAgent (ReAct)**: iterates through log search → metric queries → runbook lookup to arrive at a root cause with evidence
- **PostmortemAgent (Reflection)**: produces a structured RCA report and self-critiques it against quality criteria before finalizing
- **FastAPI backend**: exposes the pipeline as a REST API, ready for frontend integration

### Agent Paradigms Demonstrated
| Agent | Paradigm | Chapter Reference |
|---|---|---|
| TriageAgent | Plan-and-Solve | Chapter 4 |
| InvestigationAgent | ReAct | Chapter 4 |
| PostmortemAgent | Reflection | Chapter 4 |

### Challenges and Lessons
1. **Structured output parsing**: LLMs don't always return valid JSON/Python lists. Robust parsing with fallback logic is essential.
2. **Tool design**: Tool descriptions must be precise — ambiguous descriptions cause the ReAct agent to call the wrong tool.
3. **Context window management**: Passing the full investigation history in each ReAct turn grows quickly. In production, use a sliding window or summarization.
4. **Reflection threshold**: The Reflection pattern only adds value if the critique is rigorous. Vague critique prompts produce vague improvements.

### Future Plans
- [ ] **SSE streaming**: stream agent reasoning steps to a frontend in real-time
- [ ] **Vue/React frontend**: incident selector UI + live reasoning trace + markdown report viewer
- [ ] **Real log ingestion**: connect to actual log aggregators (Loki, CloudWatch, Datadog)
- [ ] **Memory across incidents**: vector store for past RCA reports to accelerate future investigations
- [ ] **Automated runbook execution**: give the agent permission to run safe remediation commands (kubectl, SQL index creation)